# Qdrant Push Embeddings


In [ ]:
%pip install qdrant_client "qdrant-client[fastembed]" fastembed chonkie

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.9/389.9 kB 38.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.5/108.5 kB 13.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.4/240.4 kB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 392.3/392.3 kB 30.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 128.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.8/324.8 kB 35.7 MB/s eta 0:00:00


## Импорты и базовая конфигурация
Ниже импортируются библиотеки, задаются пространства параметров и базовые dataclass-структуры для экспериментов.

In [ ]:
import itertools
import json
import math
import os
import random
import re
import time
import uuid
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
from fastembed import SparseTextEmbedding
from sentence_transformers import SparseEncoder
from qdrant_client import QdrantClient, models
from tqdm.auto import tqdm

print('Hybrid RAG experiment environment ready')

Hybrid RAG experiment environment ready


In [ ]:
from sentence_transformers import SentenceTransformer
from openai import OpenAI
# import faiss
from datasets import load_dataset
import torch


In [ ]:
# ---------- Search space and run configuration ----------

SERIALIZATION_CHOICES = ['markdown', 'json', 'sentence', 'html']
CHUNKING_CHOICES = ['row_level', 'table_level', 'schema_cell', 'semantic_groups']
INDEXING_CHOICES = ['sparse_only', 'dense_only', 'hybrid', 'late_interaction']
RETRIEVAL_CHOICES = ['rrf', 'two_stage', 'colbert_maxsim', 'samoe']
QUERY_EXPANSION_CHOICES = ['none', 'dual', 'llm_generated']
REASONING_CHOICES = ['direct', 'dater', 'chain_of_table', 'sql_generation']
PROMPT_CHOICES = ['minimal', 'instructed', 'few_shot', 'structured']

GENERATOR_MODELS = [
    "nvidia/nemotron-3-nano-30b-a3b:free",
    "openrouter/free"
]

EMBEDDING_MODELS = [
    'intfloat/multilingual-e5-large',
]

@dataclass
class ExperimentConfig:
    # Core retrieval axes
    serialization: str = 'markdown'
    chunking: str = 'row_level'
    indexing: str = 'hybrid'
    retrieval_strategy: str = 'rrf'
    query_expansion: str = 'none'

    # Tunables
    include_header: bool = True
    encoding_budget: int = 512
    top_k: int = 10
    rrf_k: int = 60
    rerank_k: int = 20
    dense_model: str = EMBEDDING_MODELS[0]
    sparse_model: str = "Qdrant/bm25"
    bm25_query_batch_size: int = 64

    # Runtime
    random_seed: int = 42
    dataset: str = 'custom'

print('Configuration classes loaded')

Configuration classes loaded


In [ ]:
def make_grid(param_grid: Dict[str, List[Any]]) -> List[ExperimentConfig]:
    """Generate all combinations of parameters from grid."""
    keys = sorted(param_grid.keys())
    combos = []
    for values in itertools.product(*(param_grid[k] for k in keys)):
        row = dict(zip(keys, values))
        combos.append(ExperimentConfig(**row))
    return combos


def normalize_cfg_value(v: Any) -> Any:
    """Normalize configuration value to Python types."""
    if pd.isna(v):
        return None

    if isinstance(v, str):
        s = v.strip()
        low = s.lower()
        if low == 'true':
            return True
        if low == 'false':
            return False
        try:
            return int(s)
        except ValueError:
            pass
        try:
            fv = float(s)
            return int(fv) if fv.is_integer() else fv
        except ValueError:
            return s

    if isinstance(v, (np.integer, int)):
        return int(v)
    if isinstance(v, (np.floating, float)):
        fv = float(v)
        return int(fv) if fv.is_integer() else fv
    return v


def config_to_key(cfg: ExperimentConfig) -> tuple:
    """Convert config to normalized key tuple."""
    fields = list(ExperimentConfig.__dataclass_fields__.keys())
    return tuple(normalize_cfg_value(getattr(cfg, f)) for f in fields)


def row_to_key(row: pd.Series) -> tuple:
    """Convert DataFrame row to normalized key tuple."""
    fields = list(ExperimentConfig.__dataclass_fields__.keys())
    return tuple(normalize_cfg_value(row.get(f)) for f in fields)


def exclude_completed_from_grid(
    grid: List[ExperimentConfig],
    results_csv_path: str,
    include_error_as_completed: bool = True,
) -> List[ExperimentConfig]:
    """Remove configs from grid that already exist in results CSV.

    By default, excludes both 'ok' and 'error' statuses to avoid retrying failures.
    Set include_error_as_completed=False to keep only successfully completed runs.
    """
    p = Path(results_csv_path)
    if not p.exists():
        print(f'[WARN] Results file not found: {p}. Returning original grid ({len(grid)} configs).')
        return grid

    df = pd.read_csv(p)
    if df.empty:
        print('[INFO] Results CSV is empty. Nothing to exclude.')
        return grid

    if 'status' in df.columns:
        if include_error_as_completed:
            df = df[df['status'].isin(['ok', 'error'])]
        else:
            df = df[df['status'] == 'ok']

    completed_keys = set(row_to_key(r) for _, r in df.iterrows())

    remaining = [cfg for cfg in grid if config_to_key(cfg) not in completed_keys]
    skipped = len(grid) - len(remaining)

    print(
        f'Loaded {len(df)} completed rows from {p}. '
        f'Removed {skipped} configs from grid. Remaining: {len(remaining)}'
    )
    return remaining

In [ ]:
# ============================================================================
# Data structures and serialization/chunking
# ============================================================================


@dataclass
class QueryItem:
    """Query with associated metadata."""
    qid: str
    question: str
    answer: Any
    table_id: str
    supporting_rows: Optional[List[int]] = None


@dataclass
class Chunk:
    """Text chunk with metadata and table references."""
    chunk_id: str
    table_id: str
    text: str
    row_ids: List[int]
    metadata: Dict[str, Any]


def normalize_cell(x: Any) -> str:
    """Normalize cell value to string."""
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return ''
    return str(x).strip()


def _infer_column_type(series: pd.Series, sample_size: int = 5000) -> str:
    """Infer semantic column type: number, date, category, or string."""
    non_null = series.dropna()
    if non_null.empty:
        return 'string'

    if len(non_null) > sample_size:
        non_null = non_null.iloc[:sample_size]

    if pd.api.types.is_numeric_dtype(series):
        return 'number'
    if pd.api.types.is_datetime64_any_dtype(series):
        return 'date'

    as_num = pd.to_numeric(non_null, errors='coerce')
    as_dt = pd.to_datetime(non_null, errors='coerce', utc=False, format='mixed')

    if float(as_num.notna().mean()) >= 0.95:
        return 'number'
    if float(as_dt.notna().mean()) >= 0.95:
        return 'date'

    uniq_ratio = float(non_null.nunique(dropna=True) / max(len(non_null), 1))
    return 'category' if uniq_ratio <= 0.5 else 'string'


def _top_examples(series: pd.Series, top_k: int = 3) -> List[str]:
    """Get top-k most frequent normalized non-empty values."""
    s = series.astype('string').fillna('').str.strip()
    s = s[s != '']
    if s.empty:
        return []
    return s.value_counts(dropna=False).head(top_k).index.astype(str).tolist()


def _build_schema_entries(df: pd.DataFrame) -> List[Dict[str, Any]]:
    """Build per-column schema entries with stats/examples."""
    schema_entries: List[Dict[str, Any]] = []

    for col in df.columns:
        series = df[col]
        col_type = _infer_column_type(series)
        entry: Dict[str, Any] = {
            'column_name': col,
            'dtype': col_type,
        }

        if col_type == 'number':
            numeric = pd.to_numeric(series, errors='coerce').dropna()
            entry['min'] = None if numeric.empty else float(numeric.min())
            entry['max'] = None if numeric.empty else float(numeric.max())
        elif col_type == 'date':
            dt = pd.to_datetime(series, errors='coerce', utc=False).dropna()
            entry['min'] = None if dt.empty else dt.min().isoformat()
            entry['max'] = None if dt.empty else dt.max().isoformat()
        else:
            entry['cell_examples'] = _top_examples(series, top_k=3)

        schema_entries.append(entry)

    return schema_entries


def _format_schema_text(schema_entries: List[Dict[str, Any]]) -> str:
    """Build human-readable schema text without JSON serialization."""
    lines: List[str] = []
    for e in schema_entries:
        if e['dtype'] in ('number', 'date'):
            lines.append(
                f"{e['column_name']} ({e['dtype']}): min={e.get('min')}, max={e.get('max')}"
            )
        else:
            ex = e.get('cell_examples', [])
            lines.append(
                f"{e['column_name']} ({e['dtype']}): examples={', '.join(ex) if ex else '-'}"
            )
    return 'SCHEMA:\n' + '\n'.join(lines)


def _build_cell_pairs(table_id: str, df: pd.DataFrame, encoding_budget: int) -> List[Chunk]:
    """Build unique (column, value) chunks with frequency-aware truncation."""
    budget = max(int(encoding_budget), 0)
    if budget == 0 or df.empty:
        return []

    melted = df.melt(var_name='column_name', value_name='cell_value', ignore_index=False)
    melted['cell_value'] = melted['cell_value'].astype('string').fillna('').str.strip()
    melted = melted[melted['cell_value'] != '']

    if melted.empty:
        return []

    pair_counts = (
        melted.groupby(['column_name', 'cell_value'], sort=False)
        .size()
        .rename('freq')
        .reset_index()
        .sort_values(['freq', 'column_name', 'cell_value'], ascending=[False, True, True], kind='mergesort')
    )

    if len(pair_counts) > budget:
        pair_counts = pair_counts.head(budget)

    chunks: List[Chunk] = []
    for i, row in enumerate(pair_counts.itertuples(index=False)):
        chunks.append(Chunk(
            chunk_id=f'{table_id}::cellpair::{i}',
            table_id=table_id,
            text=f"{row.column_name}={row.cell_value}",
            row_ids=[],
            metadata={
                'chunking': 'schema_cell',
                'part': 'cell',
                'column_name': row.column_name,
                'cell_value': str(row.cell_value),
                'frequency': int(row.freq),
            }
        ))

    return chunks


def serialize_row(row: pd.Series, headers: List[str], mode: str) -> str:
    """Serialize table row to string in specified format."""
    pairs = [(h, normalize_cell(row[h])) for h in headers]

    if mode == 'markdown':
        df1 = pd.DataFrame([dict(pairs)])
        return df1.to_markdown(index=False)

    if mode == 'json':
        return json.dumps(dict(pairs), ensure_ascii=False)

    if mode == 'sentence':
        return '; '.join([f'{k} - {v}' for k, v in pairs])

    if mode == 'html':
        tds = ''.join([f'<td><b>{k}</b>: {v}</td>' for k, v in pairs])
        return f'<table><tr>{tds}</tr></table>'

    raise ValueError(f'Unknown serialization mode: {mode}')


def serialize_table(df: pd.DataFrame, mode: str) -> str:
    """Serialize entire table to string in specified format."""
    if mode == 'markdown':
        return df.to_markdown(index=False)
    if mode == 'json':
        recs = df.to_dict(orient='records')
        return json.dumps(recs, ensure_ascii=False)
    if mode == 'sentence':
        headers = list(df.columns)
        lines = [serialize_row(r, headers, mode='sentence') for _, r in df.iterrows()]
        return '\n'.join(lines)
    if mode == 'html':
        return df.to_html(index=False)
    raise ValueError(f'Unknown serialization mode: {mode}')


def chunk_table(table_id: str, df: pd.DataFrame, cfg: ExperimentConfig) -> List[Chunk]:
    """Chunk table into smaller units based on configuration."""
    chunks: List[Chunk] = []
    headers = list(df.columns)

    if cfg.chunking == 'row_level':
        for ridx, (_, row) in enumerate(df.iterrows()):
            text = serialize_row(row, headers, mode=cfg.serialization)
            if cfg.include_header and cfg.serialization != 'html':
                text = 'Columns: ' + ', '.join(headers) + '\n' + text
            chunks.append(Chunk(
                chunk_id=f'{table_id}::row::{ridx}',
                table_id=table_id,
                text=text,
                row_ids=[ridx],
                metadata={'chunking': 'row_level'}
            ))
        return chunks

    if cfg.chunking == 'table_level':
        text = serialize_table(df, mode=cfg.serialization)
        chunks.append(Chunk(
            chunk_id=f'{table_id}::table',
            table_id=table_id,
            text=text,
            row_ids=list(range(len(df))),
            metadata={'chunking': 'table_level'}
        ))
        return chunks

    if cfg.chunking == 'schema_cell':
        schema_entries = _build_schema_entries(df)
        chunks.append(Chunk(
            chunk_id=f'{table_id}::schema',
            table_id=table_id,
            text=_format_schema_text(schema_entries),
            row_ids=[],
            metadata={
                'chunking': 'schema_cell',
                'part': 'schema',
                'schema': schema_entries,
            }
        ))

        chunks.extend(_build_cell_pairs(table_id=table_id, df=df, encoding_budget=cfg.encoding_budget))
        return chunks

    if cfg.chunking == 'semantic_groups':
        # Placeholder: currently fallback to table-level chunking
        text = serialize_table(df, mode=cfg.serialization)
        chunks.append(Chunk(
            chunk_id=f'{table_id}::semantic_stub',
            table_id=table_id,
            text=text,
            row_ids=list(range(len(df))),
            metadata={'chunking': 'semantic_groups', 'stub': True}
        ))
        return chunks

    raise ValueError(f'Unknown chunking mode: {cfg.chunking}')

## Query Expansion (OpenRouter)
Ячейка ниже генерирует schema/cell query-варианты из текста вопроса через OpenRouter и объединяет их с исходным запросом.

In [ ]:
UNIFIED_QUERY_EXPANSION_PROMPT = """Given a large table, I want to answer a question: {query}

Please provide TWO things in one response:
1. A list of column names that might contain necessary data
2. A list of keywords that might appear in table cells (categorical, from the question)

Answer ONLY with this JSON structure:
{{
  "columns": ["column1", "column2"],
  "keywords": ["keyword1", "keyword2"]
}}

Ответь на языке вопроса."""


def normalize_question_text(text: str) -> str:
    """Normalize question text by stripping and collapsing whitespace."""
    if not text:
        return ''
    return ' '.join(str(text).strip().split())


def _is_numeric_like(value: str) -> bool:
    """Heuristic check for numeric-like token."""
    s = value.strip().replace(',', '')
    if not s:
        return False
    return bool(re.fullmatch(r'[+-]?\d+(\.\d+)?', s))


def _build_openrouter_client() -> Optional[OpenAI]:
    """Create OpenRouter client via OpenAI SDK."""
    api_key = os.getenv('OPENROUTER_API_KEY')
    if not api_key:
        return None

    return OpenAI(
        api_key=api_key,
        base_url=os.getenv('OPENROUTER_BASE_URL', 'https://openrouter.ai/api/v1'),
    )


def _openrouter_unified_expansion(
    question: str,
    llm_client: Optional[OpenAI] = None,
    llm_model: Optional[str] = None,
    timeout: int = 60,
) -> Tuple[List[str], List[str]]:
    """Single unified OpenRouter request for both schema columns and cell keywords."""
    base = normalize_question_text(question)
    if not base:
        return [], []

    client = llm_client or _build_openrouter_client()
    if client is None:
        return [], []

    model_name = llm_model or os.getenv('OPENROUTER_MODEL') or GENERATOR_MODELS[0]
    prompt = UNIFIED_QUERY_EXPANSION_PROMPT.format(query=base)

    try:
        resp = client.chat.completions.create(
            model=model_name,
            messages=[{'role': 'user', 'content': prompt}],
            temperature=0,
            timeout=timeout,
        )
        text = ''
        if getattr(resp, 'choices', None):
            msg = getattr(resp.choices[0], 'message', None)
            text = getattr(msg, 'content', '') or ''
        result = json.loads(text) if text else {}
    except Exception as e:
        print(f'[WARN] OpenRouter unified expansion failed: {e}')
        return [], []

    if not isinstance(result, dict):
        return [], []

    # Extract and normalize columns
    columns = result.get('columns', [])
    if not isinstance(columns, list):
        columns = []

    columns_out: List[str] = []
    seen = set()
    for col in columns:
        v = normalize_question_text(str(col))
        if v and v not in seen:
            seen.add(v)
            columns_out.append(v)

    # Extract, normalize, and filter keywords
    keywords = result.get('keywords', [])
    if not isinstance(keywords, list):
        keywords = []

    keywords_out: List[str] = []
    seen = set()
    base_lower = base.lower()
    for kw in keywords:
        v = normalize_question_text(str(kw))
        # Filter: keyword must be in question and not numeric-like
        if v and v not in seen and v.lower() in base_lower and not _is_numeric_like(v):
            seen.add(v)
            keywords_out.append(v)

    return columns_out, keywords_out


def get_query_variants(
    question: str,
    query_expansion: str,
    llm_client: Optional[OpenAI] = None,
    llm_model: Optional[str] = None,
    timeout: int = 60,
) -> List[str]:
    """Generate query variants from user question semantics only (without table context)."""
    base = normalize_question_text(question)
    if not base:
        return []

    variants = [base]
    stem = base.rstrip(' ?!.')

    if query_expansion == 'dual':
        variants.append(f"table lookup: {stem}")
    elif query_expansion == 'llm_generated':
        # Single unified request instead of two separate ones
        schema_columns, cell_keywords = _openrouter_unified_expansion(
            question=base,
            llm_client=llm_client,
            llm_model=llm_model,
            timeout=timeout,
        )

        if schema_columns:
            variants.append(
                f"schema columns: {', '.join(schema_columns)} | question: {base}"
            )
        if cell_keywords:
            variants.append(
                f"cell keywords: {', '.join(cell_keywords)} | question: {base}"
            )

    uniq: List[str] = []
    seen = set()
    for v in variants:
        vv = normalize_question_text(v)
        if vv and vv not in seen:
            seen.add(vv)
            uniq.append(vv)
    return uniq

In [ ]:
from collections import defaultdict


def exp_to_name(config: ExperimentConfig, is_query: bool = False) -> str:
    """Convert experiment config to a unique name."""
    if is_query:
        return f"{config.indexing}_query_{config.query_expansion}"
    return f"{config.indexing}_{config.serialization}_{config.chunking}_{config.encoding_budget}"


def unique_by_corpus_name(
    items: List[Tuple[str, ExperimentConfig]]
) -> List[Tuple[str, ExperimentConfig]]:
    """Keep only unique configs by corpus name.

    Query expansion affects query-side pipeline, not corpus embeddings/upsert.
    """
    uniq: Dict[str, Tuple[str, ExperimentConfig]] = {}
    for name, cfg in items:
        uniq[name] = (name, cfg)
    return list(uniq.values())


def unique_by_query_name(
    items: List[Tuple[str, ExperimentConfig]]
) -> List[Tuple[str, ExperimentConfig]]:
    """Keep only unique configs by query name."""
    uniq: Dict[str, Tuple[str, ExperimentConfig]] = {}
    for _, cfg in items:
        q_name = exp_to_name(cfg, is_query=True)
        uniq[q_name] = (q_name, cfg)
    return list(uniq.values())


def get_groups_exp(
    configs: List[ExperimentConfig],
) -> Dict[str, List[Tuple[str, ExperimentConfig]]]:
    """Group experiment configs by dense model, sparse model, and dataset."""
    groups: Dict[str, List[Tuple[str, ExperimentConfig]]] = defaultdict(list)
    for cfg in configs:
        key = f"{cfg.dense_model}_{cfg.sparse_model}_{cfg.dataset}"
        groups[key].append((exp_to_name(cfg), cfg))
    return groups

In [ ]:
# ============================================================================
# Dataset adapters
# ============================================================================


def table_to_dataframe(tbl: Any) -> pd.DataFrame:
    """Convert table payloads to a pandas DataFrame."""
    if isinstance(tbl, pd.DataFrame):
        return tbl.copy()

    if isinstance(tbl, str):
        try:
            tbl = json.loads(tbl)
        except Exception:
            raise ValueError('Unsupported table string format')

    if isinstance(tbl, dict):
        header = tbl.get('header', [])
        rows = tbl.get('rows', [])

        if not isinstance(header, list):
            header = list(header) if header is not None else []
        if not isinstance(rows, list):
            rows = []

        normalized_rows: List[List[Any]] = []
        header_len = len(header)
        for row in rows:
            if isinstance(row, list):
                row_values = list(row[:header_len])
                if len(row_values) < header_len:
                    row_values.extend([None] * (header_len - len(row_values)))
            elif isinstance(row, dict):
                row_values = [row.get(col) for col in header]
            else:
                row_values = [row] + [None] * max(header_len - 1, 0)
            normalized_rows.append(row_values)

        return pd.DataFrame(normalized_rows, columns=header)

    if isinstance(tbl, list):
        return pd.DataFrame(tbl)

    raise ValueError('Unsupported table format')


def _table_array_to_dataframe(table_array: Any) -> pd.DataFrame:
    """Convert FeTaQA-like table_array into a DataFrame with header row."""
    if not isinstance(table_array, list) or not table_array:
        raise ValueError('Unsupported table_array format')

    header = table_array[0]
    rows = table_array[1:]
    if not isinstance(header, list):
        raise ValueError('table_array must start with a header row')

    normalized_rows: List[List[Any]] = []
    header_len = len(header)
    for row in rows:
        if isinstance(row, list):
            row_values = list(row[:header_len])
            if len(row_values) < header_len:
                row_values.extend([None] * (header_len - len(row_values)))
            normalized_rows.append(row_values)
        elif isinstance(row, dict):
            normalized_rows.append([row.get(col) for col in header])
        else:
            normalized_rows.append([row] + [None] * max(header_len - 1, 0))

    return pd.DataFrame(normalized_rows, columns=header)


def _normalize_answers(value: Any) -> List[str]:
    """Normalize answer payload to a non-empty list of strings."""
    if value is None:
        return []

    if isinstance(value, str):
        cleaned = normalize_question_text(value)
        return [cleaned] if cleaned else []

    if isinstance(value, (list, tuple, set)):
        answers: List[str] = []
        for item in value:
            cleaned = normalize_question_text(item)
            if cleaned:
                answers.append(cleaned)
        return answers

    cleaned = normalize_question_text(value)
    return [cleaned] if cleaned else []


def _highlighted_cell_ids_to_rows(highlighted_cell_ids: Any) -> Optional[List[int]]:
    """Convert FeTaQA highlighted cell pairs like [[row, col], ...] to row ids."""
    if not isinstance(highlighted_cell_ids, list) or not highlighted_cell_ids:
        return None

    row_ids: List[int] = []
    for item in highlighted_cell_ids:
        if isinstance(item, (list, tuple)) and item:
            row_id = item[0]
        else:
            row_id = item

        if isinstance(row_id, bool):
            continue
        try:
            row_ids.append(int(row_id))
        except Exception:
            continue

    return row_ids or None


def _open_wikitable_row_to_dataframe(row: pd.Series) -> pd.DataFrame:
    """Convert one Open WikiTable row into a DataFrame."""
    header = row.get('header', [])
    rows = row.get('rows', [])

    if isinstance(header, str):
        try:
            header = json.loads(header)
        except Exception:
            header = [header]
    if isinstance(rows, str):
        try:
            rows = json.loads(rows)
        except Exception:
            rows = []

    if not isinstance(header, list):
        header = list(header) if header is not None else []
    if not isinstance(rows, list):
        rows = []

    normalized_rows: List[List[Any]] = []
    header_len = len(header)
    for item in rows:
        if isinstance(item, list):
            row_values = list(item[:header_len])
            if len(row_values) < header_len:
                row_values.extend([None] * (header_len - len(row_values)))
        elif isinstance(item, dict):
            row_values = [item.get(col) for col in header]
        else:
            row_values = [item] + [None] * max(header_len - 1, 0)
        normalized_rows.append(row_values)

    return pd.DataFrame(normalized_rows, columns=header)


def _open_wikitable_idx_to_rows(value: Any) -> Optional[List[int]]:
    """Convert Open WikiTable positive index payload to row ids."""
    if not isinstance(value, list) or not value:
        return None

    row_ids: List[int] = []
    for item in value:
        if isinstance(item, (list, tuple)) and item:
            row_id = item[0]
        else:
            row_id = item

        if isinstance(row_id, bool):
            continue
        try:
            row_ids.append(int(row_id))
        except Exception:
            continue

    return row_ids or None


def load_custom_fetaqa_like(
    base_questions: str = 'rag_questions',
    data_dir: str = 'data',
    limit: Optional[int] = 200,
    sample_random: bool = True,
    random_seed: int = 42,
) -> tuple[Dict[str, pd.DataFrame], List[QueryItem]]:
    """Load custom FeTaQA-like dataset from JSON and CSV files."""
    base_questions = Path(base_questions)
    data_dir = Path(data_dir)

    tables: Dict[str, pd.DataFrame] = {}
    queries: List[QueryItem] = []

    paths = sorted(
        [p for p in base_questions.rglob('*.json') if not p.name.endswith('.error.json')]
    )
    total_files = len(paths)
    if limit and total_files > limit:
        if sample_random:
            rng = random.Random(random_seed)
            paths = sorted(rng.sample(paths, k=limit))
            print(f'Randomly sampled {limit} files from {total_files} using seed={random_seed}')
        else:
            paths = paths[:limit]

    print(f'Loading {len(paths)} custom dataset files...')
    for jp in paths:
        rel = jp.relative_to(base_questions)
        csv_path = (data_dir / rel).with_suffix('.csv')
        table_id = str(rel.with_suffix(''))

        if csv_path.exists() and table_id not in tables:
            try:
                tables[table_id] = pd.read_csv(csv_path, sep='|', index_col=0).reset_index(drop=True)
            except Exception:
                continue

        try:
            obj = json.loads(jp.read_text(encoding='utf-8'))
        except Exception:
            continue

        if not isinstance(obj, list):
            continue

        for i, q in enumerate(obj):
            if not isinstance(q, dict):
                continue
            question = normalize_question_text(q.get('question', ''))
            answer = _normalize_answers(q.get('answers') or q.get('answer'))
            supporting_rows = q.get('supporting_rows') if isinstance(q.get('supporting_rows'), list) else None
            qid = f'{table_id}::q::{i}'
            queries.append(
                QueryItem(
                    qid=qid,
                    question=question,
                    answer=answer,
                    table_id=table_id,
                    supporting_rows=supporting_rows,
                )
            )

    return tables, queries


def load_open_wikitable_from_files(
    test_path: str = 'train.json',
    tables_path: str = 'tables.json',
    limit: Optional[int] = None,
    sample_random: bool = False,
    random_seed: int = 42,
) -> tuple[Dict[str, pd.DataFrame], List[QueryItem]]:
    """Load Open WikiTable from local test and tables JSON files."""
    test_file = Path(test_path)
    tables_file = Path(tables_path)

    if not test_file.exists():
        raise FileNotFoundError(f'Missing Open WikiTable test file: {test_file}')
    if not tables_file.exists():
        raise FileNotFoundError(f'Missing Open WikiTable tables file: {tables_file}')

    test_df = pd.read_json(test_file)
    tables_df = pd.read_json(tables_file)
    test_df = test_df[test_df['dataset'] == 'WikiTQ']
    tables_df = tables_df[tables_df['dataset'] == 'WikiTQ']

    if limit and len(test_df) > limit:
        if sample_random:
            test_df = test_df.sample(n=limit, random_state=random_seed).reset_index(drop=True)
            print(f'Randomly sampled {limit} questions from {len(test_df)} Open WikiTable rows')
        else:
            test_df = test_df.head(limit).reset_index(drop=True)

    target_table_ids = set(test_df['original_table_id'].astype(str).tolist())

    tables: Dict[str, pd.DataFrame] = {}
    for _, row in tables_df.iterrows():
        table_id = str(row.get('original_table_id', ''))
        if not table_id or table_id not in target_table_ids or table_id in tables:
            continue
        tables[table_id] = _open_wikitable_row_to_dataframe(row)

    queries: List[QueryItem] = []
    for i, ex in enumerate(test_df.itertuples(index=False)):
        table_id = str(getattr(ex, 'original_table_id', ''))
        if not table_id:
            continue

        question = normalize_question_text(getattr(ex, 'question', ''))
        answer = getattr(ex, 'answer', '')
        question_id = getattr(ex, 'question_id', None)
        supporting_rows = _open_wikitable_idx_to_rows(getattr(ex, 'hard_positive_idx', None))
        if supporting_rows is None:
            supporting_rows = _open_wikitable_idx_to_rows(getattr(ex, 'positive_idx', None))

        qid = str(question_id or f'{table_id}::q::{i}')
        queries.append(
            QueryItem(
                qid=qid,
                question=question,
                answer=answer,
                table_id=table_id,
                supporting_rows=supporting_rows,
            )
        )

    missing_tables = sum(1 for q in queries if q.table_id not in tables)
    if missing_tables:
        print(f'[WARN] {missing_tables} queries reference tables not found in {tables_file}')

    print(
        f'Loaded Open WikiTable: tables={len(tables)} / {len(target_table_ids)}, '
        f'queries={len(queries)} from {test_file}'
    )
    return tables, queries


def load_hf_tabular_dataset(
    dataset_name: str, split: str = 'train[:200]'
) -> tuple[Dict[str, pd.DataFrame], List[QueryItem]]:
    """Load tabular dataset from Hugging Face, including lighteval/wikitablequestions."""
    ds = load_dataset(dataset_name, split=split)
    tables: Dict[str, pd.DataFrame] = {}
    queries: List[QueryItem] = []

    print(f'Loading {dataset_name}...')
    for i, ex in enumerate(ds):
        if not isinstance(ex, dict):
            ex = ex.to_dict() if hasattr(ex, 'to_dict') else {}

        table_payload = ex.get('table') or ex.get('table_array') or ex.get('table_data') or ex.get('context')
        if table_payload is None:
            continue

        try:
            if isinstance(table_payload, list) and table_payload and isinstance(table_payload[0], list):
                df = _table_array_to_dataframe(table_payload)
            else:
                df = table_to_dataframe(table_payload)
        except Exception:
            continue

        table_id = str(ex.get('table_id') or ex.get('feta_id') or ex.get('id') or ex.get('name') or f'{dataset_name}::{i}')
        if table_id not in tables:
            tables[table_id] = df

        question = normalize_question_text(ex.get('question') or ex.get('query') or '')
        answer = _normalize_answers(ex.get('answers') or ex.get('answer'))

        highlighted_cell_ids = ex.get('highlighted_cell_ids')
        if not isinstance(highlighted_cell_ids, list):
            highlighted_cell_ids = None

        supporting_rows = ex.get('supporting_rows')
        if not isinstance(supporting_rows, list):
            supporting_rows = _highlighted_cell_ids_to_rows(highlighted_cell_ids)

        qid = str(ex.get('qid') or ex.get('question_id') or ex.get('id') or f'{table_id}::q::{i}')
        queries.append(
            QueryItem(
                qid=qid,
                question=question,
                answer=answer,
                table_id=table_id,
                supporting_rows=supporting_rows,
            )
        )

    return tables, queries


# Select one source per run
DATASET_SOURCE = 'lighteval/wikitablequestions'  # custom | fetaqa | open_wikitable

if DATASET_SOURCE == 'custom':
    tables, queries = load_custom_fetaqa_like(
        limit=1500,
        sample_random=True,
        random_seed=42,
    )
elif DATASET_SOURCE == 'fetaqa':
    tables, queries = load_hf_tabular_dataset('DongfuJiang/FeTaQA', split='train[:1000]')
elif DATASET_SOURCE == 'open_wikitable':
    tables, queries = load_open_wikitable_from_files(sample_random=True, limit=1500)
else:
    raise ValueError(DATASET_SOURCE)

print(f'Loaded tables={len(tables)}, queries={len(queries)}')

Randomly sampled 1500 files from 5011 using seed=42
Loading 1500 custom dataset files...
Loaded tables=1500, queries=10488


## Быстрая проверка и локальная отладка chunking
Сначала проверяем качество разбиения на chunks, затем при необходимости смотрим случайную таблицу для ручной валидации.

In [ ]:
# ============================================================================
# Chunking output quick test (works for any chunking variant)
# ============================================================================

def preview_chunking_output(
    tables_obj: Optional[Dict[str, pd.DataFrame]] = None,
    table_id: Optional[str] = None,
    chunking: str = 'schema_cell',
    serialization: str = 'markdown',
    encoding_budget: int = 512,
    include_header: bool = True,
) -> List[Chunk]:
    """Run chunking once and print a compact preview of produced chunks."""
    tables_ref = tables_obj if tables_obj is not None else globals().get('tables')
    if not tables_ref:
        raise ValueError('tables is empty or not defined. Run dataset loading cell first.')

    if table_id is None:
        table_id = next(iter(tables_ref.keys()))

    if table_id not in tables_ref:
        raise KeyError(f'table_id={table_id} is not found in tables')

    df = tables_ref[table_id]
    cfg = ExperimentConfig(
        serialization=serialization,
        chunking=chunking,
        indexing='dense',
        include_header=include_header,
        encoding_budget=int(encoding_budget),
        dataset='custom',
    )

    t0 = time.perf_counter()
    chunks = chunk_table(table_id=table_id, df=df, cfg=cfg)
    dt_ms = (time.perf_counter() - t0) * 1000

    print(f'table_id={table_id} | shape={df.shape}')
    print(
        f'chunking={chunking} | serialization={serialization} | '
        f'encoding_budget={encoding_budget} | chunks={len(chunks)} | {dt_ms:.1f} ms'
    )

    if not chunks:
        print('No chunks produced.')
        return chunks

    part_counts = pd.Series([c.metadata.get('part', 'n/a') for c in chunks]).value_counts()
    print('part distribution:')
    print(part_counts.to_string())


    for i, c in enumerate(chunks, start=1):
        print(
            f'[{i}] chunk_id={c.chunk_id} | rows={len(c.row_ids)} | '
            f'meta={c.metadata} | text={c.text}'
        )

    return chunks



### Локальная отладка таблиц
Следующие 2 ячейки выбирают случайную таблицу для ручной проверки содержимого.

In [ ]:
import random
random_key, random_val = random.choice(list(tables.items()))

In [ ]:
random_key

'4153976/table_0'

In [ ]:

# Example usage:
# 1) preview_chunking_output(chunking='row_level', serialization='markdown', encoding_budget=512)
# 2) preview_chunking_output(chunking='table_level', serialization='json', encoding_budget=512)
# 3) preview_chunking_output(chunking='schema_cell', serialization='sentence', encoding_budget=128)

preview_chunks = preview_chunking_output(
    chunking='schema_cell',
    table_id='109836/table_8',
    serialization='sentence',
    encoding_budget=128,
)


### Локальная отладка вопросов

In [ ]:
import random

# Quick test: generate query variants using questions from loaded dataset
if not queries:
    raise ValueError('queries is empty. Run dataset loading cell first.')

sample_size = 5
sampled_questions = [random.choice(queries).question for q in range(sample_size)]

# test_modes = ['none', 'dual', 'llm_generated']
test_modes = ['llm_generated']

for idx, question in enumerate(sampled_questions, start=1):
    print('=' * 100)
    print(f'Question {idx}: {question}')
    for mode in test_modes:
        variants = get_query_variants(question, mode)
        print(f'\nmode={mode} -> {len(variants)} variants')
        for i, v in enumerate(variants, start=1):
            print(f'  {i}. {v}')
    print()

## Индексация и загрузка в Qdrant
Ниже идут функции подготовки векторов, создание коллекций и два этапа upsert (корпус и query-варианты).

### Подготовка функций upsert
Утилиты: slug коллекций, chunking корпуса, кодирование dense/sparse и батчевый upsert.

In [ ]:
def slugify_collection_name(name: str) -> str:
    """Convert collection name to valid Qdrant format.

    Replaces special characters with underscores, keeps only alphanumeric,
    hyphens and underscores.
    """
    clean_name = re.sub(r'[^a-zA-Z0-9\-_]', '_', name)
    return re.sub(r'_+', '_', clean_name).strip('_')


UPSERT_BATCH_SIZE = 1000


def prepare_corpus(
    cfg: ExperimentConfig, tables: Dict[str, pd.DataFrame]
) -> List[Chunk]:
    """Prepare corpus by chunking all tables."""
    chunks: List[Chunk] = []
    print(f'Chunking {len(tables)} tables...')
    for table_id, df in tables.items():
        chunks.extend(chunk_table(table_id, df, cfg))
    return chunks


def encode_texts(
    texts: List[str],
    dense_model: Optional[Any] = None,
    sparse_model: Optional[SparseEncoder | SparseTextEmbedding] = None,
    need_dense: bool = False,
    need_sparse: bool = False,
    dense_prompt_name: Optional[str] = None,
) -> tuple[Optional[Any], Optional[Any]]:
    """Encode texts using dense and/or sparse models."""
    dense_embds = None
    sparse_embds = None

    if need_dense and dense_model:
        dense_kargs = {'prompt_name': dense_prompt_name} if dense_prompt_name else {}
        gpu_count = torch.cuda.device_count()

        if gpu_count > 1:
            print(f"Обнаружено GPU: {gpu_count}. Запуск мульти-процессинга...")
            pool = dense_model.start_multi_process_pool()
            dense_embds = dense_model.encode(
                texts,
                pool=pool,
                batch_size=256,
                normalize_embeddings=True,
                show_progress_bar=True,
                **dense_kargs,
            )
            dense_model.stop_multi_process_pool(pool)
        else:
            print("Работаем на одной карте ...")
            dense_embds = dense_model.encode(
                texts,
                batch_size=1,
                normalize_embeddings=True,
                show_progress_bar=True,
                **dense_kargs
            )

    if need_sparse and sparse_model:
        if isinstance(sparse_model, SparseTextEmbedding):
            sparse_embds = [
                models.SparseVector(indices=sp_vec.indices, values=sp_vec.values)
                for sp_vec in sparse_model.embed(texts)
            ]
        else:
            pool = sparse_model.start_multi_process_pool()
            sparse_embds = [
                models.SparseVector(indices=sp_vec.coalesce().indices().tolist()[0], values=sp_vec.coalesce().values().tolist())
                for sp_vec in sparse_model.encode_document(texts, show_progress_bar=True, pool=pool)
            ]
            sparse_model.stop_multi_process_pool(pool)

    return dense_embds, sparse_embds


def build_points_with_vectors(
    items: List[tuple],
    embeddings: Dict[str, Any],
    configs: List[Tuple[str, ExperimentConfig]],
    payload_fn: callable,
    id_fn: callable,
) -> list:
    """Build PointStruct list with vectors from multiple configs.

    Args:
        items: List of items to create points from (chunks or queries)
        embeddings: Dict with 'dense' and 'sparse' embedding arrays
        configs: List of (name, config) tuples
        payload_fn: Function to create payload from item
        id_fn: Function to create point ID from item
    """
    points_by_id = {}

    # Create point skeletons with payloads
    for item in items:
        point_id = id_fn(item)
        if point_id not in points_by_id:
            points_by_id[point_id] = models.PointStruct(
                id=point_id,
                vector={},
                payload=payload_fn(item),
            )

    # Fill vectors
    dense_embds = embeddings.get('dense')
    sparse_embds = embeddings.get('sparse')

    for idx, item in enumerate(items):
        point_id = id_fn(item)
        if point_id not in points_by_id:
            continue

        for cfg_name, cfg_obj in configs:
            if cfg_obj.indexing == 'dense' and dense_embds is not None:
                vec_value = dense_embds[idx]
                if isinstance(vec_value, np.ndarray):
                    vec_value = vec_value.tolist()
                points_by_id[point_id].vector[cfg_name] = vec_value
            elif cfg_obj.indexing != 'dense' and sparse_embds is not None:
                points_by_id[point_id].vector[cfg_name] = sparse_embds[idx]

    return [p for p in points_by_id.values() if p.vector]


def _try_upsert_with_retry(
    client: QdrantClient,
    collection_name: str,
    points: List[models.PointStruct],
    max_batch: int,
) -> int:
    """Try to upsert chunk, retry with smaller batch on payload error. Returns number of points upserted."""
    current = max_batch
    while current >= 1:
        try:
            client.upsert(collection_name=collection_name, points=points[:current])
            return current
        except Exception as e:
            msg = str(e)
            is_payload_error = (
                'Payload error' in msg or 'larger than allowed' in msg or 'Bad Request' in msg
            )
            if is_payload_error and current > 1:
                current = max(1, current // 2)
            else:
                raise
    return 0


def upsert_points_in_batches(
    client: QdrantClient,
    collection_name: str,
    points: List[models.PointStruct],
    batch_size: int = UPSERT_BATCH_SIZE,
) -> None:
    """Upsert points in chunks. Auto-reduce batch size on payload limit errors."""
    if not points:
        return

    if batch_size < 1:
        raise ValueError('batch_size must be >= 1')

    total = len(points)
    with tqdm(total=total, desc=f'Upserting to {collection_name}', unit='points') as pbar:
        i = 0
        while i < total:
            remaining = points[i:]
            current_batch = min(batch_size, len(remaining))
            upserted = _try_upsert_with_retry(client, collection_name, remaining, current_batch)
            pbar.update(upserted)
            i += upserted

    print(f'Upserted {total} points into {collection_name} (batch size={batch_size})')

### Формирование grid экспериментов

In [ ]:
stage1_grid = make_grid({
    'serialization': ['markdown', 'json', 'sentence'],
    'chunking': ['table_level'], #'schema_cell'
    'dense_model': ['Qwen/Qwen3-Embedding-4B'],
    'sparse_model': ['naver/neuclir22-splade-ru'], #  , "Qdrant/bm25"
    'indexing': ['sparse', 'dense'],
    'dataset': [DATASET_SOURCE],
    'query_expansion': ['none', 'llm_generated'],
})

### Подключение к Qdrant

In [ ]:
client = QdrantClient(
    host=os.getenv('QDRANT_HOST'),
    port=6333,
    api_key=os.getenv('QDRANT_API_KEY'),
    # prefer_grpc=True,
    https=False,
    timeout=3000,
)

### Проверка коллекций и подготовка групп

In [ ]:
client.get_collections()

In [ ]:
exp_groups = get_groups_exp(stage1_grid)

In [ ]:
exp_groups.keys()

### Создание коллекций под векторные пространства

In [ ]:
exp_groups = get_groups_exp(stage1_grid)

for key, val in exp_groups.items():
    dense_model = SentenceTransformer(val[0][1].dense_model)
    vec_config = {}
    sparse_config = {}

    unique_val = unique_by_corpus_name(val)
    for name_exp in unique_val:
        if name_exp[1].indexing == 'dense':
            vec_config[name_exp[0]] = models.VectorParams(
                size=dense_model.get_sentence_embedding_dimension(),
                distance=models.Distance.COSINE
            )
        if name_exp[1].indexing == 'sparse':
            sparse_config[name_exp[0]] = models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )

    unique_query_val = unique_by_query_name(val)
    for q_name_exp in unique_query_val:
        if q_name_exp[1].indexing == 'dense':
            vec_config[q_name_exp[0]] = models.VectorParams(
                size=dense_model.get_sentence_embedding_dimension(),
                distance=models.Distance.COSINE
            )
        if q_name_exp[1].indexing == 'sparse':
            sparse_config[q_name_exp[0]] = models.SparseVectorParams(
                modifier=models.Modifier.IDF
            )

    client.create_collection(
        collection_name=slugify_collection_name(key),
        vectors_config=vec_config,
        sparse_vectors_config=sparse_config,
    )

### Upsert chunks

In [ ]:
for key, val in exp_groups.items():
    dense_model = SentenceTransformer(val[0][1].dense_model)
    sparse_model = SparseTextEmbedding("Qdrant/bm25") if val[0][1].sparse_model == "Qdrant/bm25" else SparseEncoder("naver/neuclir22-splade-ru", token=os.getenv('HF_TOKEN'))

    # Group configs by corpus characteristics
    def corpus_key(cfg: ExperimentConfig) -> tuple:
        return (cfg.serialization, cfg.chunking, cfg.encoding_budget)

    groups: Dict[tuple, List[Tuple[str, ExperimentConfig]]] = defaultdict(list)
    for cfg in val:
        groups[corpus_key(cfg[1])].append(cfg)

    # Upsert each corpus group
    for _, group_cfgs in groups.items():
        base_cfg = group_cfgs[0][1]
        chunks = prepare_corpus(base_cfg, tables)
        texts = [c.text for c in chunks]

        unique_group_cfgs = unique_by_corpus_name(group_cfgs)
        vector_names = [cfg_name for cfg_name, _ in unique_group_cfgs]

        need_dense = any(cfg[1].indexing == 'dense' for cfg in unique_group_cfgs)
        need_sparse = any(cfg[1].indexing != 'dense' for cfg in unique_group_cfgs)

        # Encode all texts
        dense_embds, sparse_embds = encode_texts(
            texts,
            dense_model=dense_model,
            sparse_model=sparse_model,
            need_dense=need_dense,
            need_sparse=need_sparse,
            # dense_prompt_name='document'
        )

        # Build points
        def chunk_payload_fn(chunk: Chunk) -> dict:
            return chunk.__dict__

        def chunk_id_fn(chunk: Chunk) -> str:
            # Keep IDs unique across serialization/chunking variants.
            raw_key = (
                f"corpus::{key}::{base_cfg.serialization}::"
                f"{base_cfg.chunking}::{base_cfg.encoding_budget}::{chunk.chunk_id}"
            )
            return str(uuid.uuid5(uuid.NAMESPACE_URL, raw_key))

        points = build_points_with_vectors(
            items=chunks,
            embeddings={'dense': dense_embds, 'sparse': sparse_embds},
            configs=unique_group_cfgs,
            payload_fn=chunk_payload_fn,
            id_fn=chunk_id_fn,
        )

        print(
            f"[{key}] corpus={base_cfg.serialization}/{base_cfg.chunking}/"
            f"{base_cfg.encoding_budget}: chunks={len(chunks)}, points={len(points)}, "
            f"vectors={vector_names}"
        )

        if points:
            upsert_points_in_batches(
                client=client,
                collection_name=slugify_collection_name(key),
                points=points,
                batch_size=UPSERT_BATCH_SIZE,
            )

### Upsert query-вариантов

In [ ]:
# ========== Query-variant upsert ==========
def load_query_variants(dataset: str, query_expansion: str) -> Dict[str, List[str]]:
    """Load pre-generated query variants from JSON file.

    Returns empty dict if file is missing or invalid.
    """
    filepath = Path('query_variants') / f'{dataset}_{query_expansion}.json'
    if not filepath.exists():
        print(f'[INFO] Query variants file not found: {filepath}')
        return {}

    try:
        with open(filepath, 'r', encoding='utf-8') as f:
            data = json.load(f)

        variant_map: Dict[str, List[str]] = {}
        for q_item in data.get('queries', []):
            if not isinstance(q_item, dict):
                continue
            q_text = q_item.get('question_original')
            variants = q_item.get('variants', [])
            if q_text and isinstance(variants, list):
                cleaned = [normalize_question_text(v) for v in variants if normalize_question_text(v)]
                if cleaned:
                    variant_map[q_text] = cleaned

        print(f'[✓] Loaded {len(variant_map)} question variants from {filepath}')
        return variant_map
    except Exception as e:
        print(f'[ERROR] Failed to load query variants from {filepath}: {e}')
        return {}


for key, val in exp_groups.items():
    dense_model = SentenceTransformer(val[0][1].dense_model)
    sparse_model = SparseTextEmbedding("Qdrant/bm25") if val[0][1].sparse_model == "Qdrant/bm25" else SparseEncoder("naver/neuclir22-splade-ru", token=os.getenv('HF_TOKEN'))

    unique_query_cfgs = unique_by_query_name(val)

    # Group by expansion mode
    query_cfgs_by_expansion: Dict[
        str, List[Tuple[str, ExperimentConfig]]
    ] = defaultdict(list)
    for q_name, q_cfg in unique_query_cfgs:
        query_cfgs_by_expansion[q_cfg.query_expansion].append((q_name, q_cfg))

    for expansion_mode, cfg_items in tqdm(
        query_cfgs_by_expansion.items(), desc='Query expansions'
    ):
        dataset_name = val[0][1].dataset
        loaded_variant_map = load_query_variants(dataset_name, expansion_mode)

        # Per-question fallback: generate only when variant is missing for this specific question.
        variant_rows = []
        generated_count = 0
        loaded_count = 0

        for q in tqdm(queries, desc=f'Preparing {expansion_mode}', leave=False):
            variants = loaded_variant_map.get(q.question)
            if variants:
                loaded_count += 1
            else:
                variants = get_query_variants(q.question, expansion_mode)
                generated_count += 1

            if not variants:
                variants = [q.question]

            for vidx, vtext in enumerate(variants):
                variant_rows.append((q, vidx, vtext))

        print(
            f"[{key}] expansion={expansion_mode}: "
            f"loaded_questions={loaded_count}, generated_questions={generated_count}"
        )

        if not variant_rows:
            print(f"[{key}] expansion={expansion_mode}: no query variants")
            continue

        query_texts = [row[2] for row in variant_rows]
        need_dense = any(cfg.indexing == 'dense' for _, cfg in cfg_items)
        need_sparse = any(cfg.indexing != 'dense' for _, cfg in cfg_items)

        # Encode query texts
        dense_query_embds, sparse_query_embds = encode_texts(
            query_texts,
            dense_model=dense_model,
            sparse_model=sparse_model,
            need_dense=need_dense,
            need_sparse=need_sparse,
            dense_prompt_name='query'
        )

        # Build points
        def query_payload_fn(variant_row: tuple) -> dict:
            q, vidx, vtext = variant_row
            return {
                'record_type': 'query_variant',
                'qid': q.qid,
                'table_id': q.table_id,
                'question_original': q.question,
                'question_variant': vtext,
                'query_expansion': expansion_mode,
                'variant_index': vidx,
                'supporting_rows': q.supporting_rows,
            }

        def query_id_fn(variant_row: tuple) -> str:
            q, vidx, vtext = variant_row
            raw_key = f"query::{key}::{expansion_mode}::{q.qid}::{vidx}::{vtext}"
            return str(uuid.uuid5(uuid.NAMESPACE_URL, raw_key))

        q_points = build_points_with_vectors(
            items=variant_rows,
            embeddings={'dense': dense_query_embds, 'sparse': sparse_query_embds},
            configs=cfg_items,
            payload_fn=query_payload_fn,
            id_fn=query_id_fn,
        )

        vector_names = [q_name for q_name, _ in cfg_items]

        print(
            f"[{key}] expansion={expansion_mode}: unique_points={len(q_points)}, "
            f"vectors={vector_names}"
        )

        if q_points:
            upsert_points_in_batches(
                client=client,
                collection_name=slugify_collection_name(key),
                points=q_points,
                batch_size=UPSERT_BATCH_SIZE,
            )

## Baseline gen

In [ ]:
# ---------- Baseline collection: Chonkie recursive markdown chunker + dense vectors ----------
BASELINE_DENSE_MODEL = 'Qwen/Qwen3-8B'
BASELINE_QUERY_EXPANSION = 'none'
BASELINE_ENCODING_BUDGET = 512  # naming compatibility with evaluation notebook

from chonkie import RecursiveChunker


def _extract_chunk_text(item: Any) -> str:
    if item is None:
        return ''
    if isinstance(item, str):
        return item
    text_attr = getattr(item, 'text', None)
    if isinstance(text_attr, str):
        return text_attr
    if isinstance(item, dict):
        for k in ('text', 'content', 'chunk', 'value'):
            v = item.get(k)
            if isinstance(v, str):
                return v
    return str(item)


def chonkie_markdown_chunks(
    text: str,
) -> List[str]:
    text = (text or '').strip()
    if not text:
        return []

    chunker = RecursiveChunker.from_recipe("markdown", lang="en")
    parts_raw = chunker.chunk(text)
    parts = [_extract_chunk_text(x).strip() for x in parts_raw]
    return [p for p in parts if p]


baseline_dataset = DATASET_SOURCE
baseline_key = f'{BASELINE_DENSE_MODEL}_{baseline_dataset}_baseline_recursive_dense_markdown'
baseline_collection = slugify_collection_name(baseline_key)
baseline_corpus_vector_name = f'dense_markdown_recursive_{BASELINE_ENCODING_BUDGET}'
baseline_query_vector_name = f'dense_query_{BASELINE_QUERY_EXPANSION}'

print('Baseline collection:', baseline_collection)
print('Vectors:', baseline_corpus_vector_name, baseline_query_vector_name)

dense_model = SentenceTransformer(BASELINE_DENSE_MODEL)
dense_size = dense_model.get_sentence_embedding_dimension()

if not client.collection_exists(baseline_collection):
    client.create_collection(
        collection_name=baseline_collection,
        vectors_config={
            baseline_corpus_vector_name: models.VectorParams(
                size=dense_size,
                distance=models.Distance.COSINE,
            ),
            baseline_query_vector_name: models.VectorParams(
                size=dense_size,
                distance=models.Distance.COSINE,
            ),
        },
    )
    print('[✓] Created collection')
else:
    print('[i] Collection already exists, appending/overwriting points by deterministic ids')

# 1) Corpus upsert with Chonkie recursive markdown chunking.
baseline_chunks: List[Chunk] = []
for table_id, df in tqdm(tables.items(), desc='Chonkie recursive markdown chunking'):
    table_md = serialize_table(df, mode='markdown')
    parts = chonkie_markdown_chunks(
        table_md,
    )
    for idx, part in enumerate(parts):
        baseline_chunks.append(
            Chunk(
                chunk_id=f'{table_id}::recursive::{idx}',
                table_id=table_id,
                text=part,
                row_ids=list(range(len(df))),
                metadata={
                    'chunking': 'recursive_markdown',
                    'chunker_backend': 'chonkie',
                    'serialization': 'markdown',
                    'encoding_budget': BASELINE_ENCODING_BUDGET,
                },
            )
        )

corpus_texts = [c.text for c in baseline_chunks]
corpus_embds = dense_model.encode(
    corpus_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
) if corpus_texts else []

corpus_points: List[models.PointStruct] = []
for i, c in enumerate(baseline_chunks):
    raw_id = (
        f'baseline::corpus::{baseline_collection}::'
        f'{c.table_id}::{c.chunk_id}'
    )
    pid = str(uuid.uuid5(uuid.NAMESPACE_URL, raw_id))
    payload = {
        **c.__dict__,
        'record_type': 'corpus_chunk',
    }
    vec = corpus_embds[i].tolist() if isinstance(corpus_embds[i], np.ndarray) else corpus_embds[i]
    corpus_points.append(
        models.PointStruct(
            id=pid,
            vector={baseline_corpus_vector_name: vec},
            payload=payload,
        )
    )

upsert_points_in_batches(
    client=client,
    collection_name=baseline_collection,
    points=corpus_points,
    batch_size=UPSERT_BATCH_SIZE,
)

# 2) Query upsert for query_expansion='none' (needed by evaluation notebook).
query_texts = [normalize_question_text(q.question) for q in queries]
query_embds = dense_model.encode(
    query_texts,
    batch_size=64,
    normalize_embeddings=True,
    show_progress_bar=True,
) if query_texts else []

query_points: List[models.PointStruct] = []
for i, q in enumerate(queries):
    qtext = query_texts[i]
    raw_id = f'baseline::query::{baseline_collection}::{q.qid}::0::{qtext}'
    pid = str(uuid.uuid5(uuid.NAMESPACE_URL, raw_id))
    payload = {
        'record_type': 'query_variant',
        'qid': q.qid,
        'table_id': q.table_id,
        'question_original': q.question,
        'question_variant': qtext,
        'query_expansion': BASELINE_QUERY_EXPANSION,
        'variant_index': 0,
        'supporting_rows': q.supporting_rows,
    }
    vec = query_embds[i].tolist() if isinstance(query_embds[i], np.ndarray) else query_embds[i]
    query_points.append(
        models.PointStruct(
            id=pid,
            vector={baseline_query_vector_name: vec},
            payload=payload,
        )
    )

upsert_points_in_batches(
    client=client,
    collection_name=baseline_collection,
    points=query_points,
    batch_size=UPSERT_BATCH_SIZE,
)

print(
    f'[✓] Baseline ready: collection={baseline_collection} | '
    f'corpus_points={len(corpus_points)} | query_points={len(query_points)}'
)